In [1]:
import uproot
import pandas as pd

# ─────────────────────────────────────────────────────────────────────
# Load the training ntuple into a pandas DataFrame.
# Each row is one FD π±/K± track passing the script's filters.
# Columns are documented in the script's startup banner (53 total) and
# in the column-map table in Section 4e.
# ─────────────────────────────────────────────────────────────────────
#df = uproot.open(
 #   "/volatile/clas12/cooperb/SULI/pid_training_test.root:PhysicsEvents"
    #"/volatile/clas12/cooperb/SULI/pid_training_v1.root:PhysicsEvents"
#).arrays(library="pd")

cols = ["pid", "mc_matching_pid", "p", "theta", "beta", "chi2pid", "rich_RQ", "vz"]
df = uproot.open("/volatile/clas12/cooperb/SULI/pid_training_v1.root:PhysicsEvents").arrays(cols, library="pd")





In [2]:
# Basic sanity check: how many tracks, how many features?
# Expect ~1-2 million rows from one HIPO file, exactly 53 columns.
print("Shape:", df.shape)

# ─────────────────────────────────────────────────────────────────────
# What did the Event Builder reconstruct each track as?
# Column "pid" is the EB-assigned PDG code, filtered by the script to
# {211, -211, 321, -321} (π+, π-, K+, K-).
# ─────────────────────────────────────────────────────────────────────
print("\nEB-assigned PID counts:")
print(df["pid"].value_counts())

Shape: (2724581, 8)

EB-assigned PID counts:
pid
211     1505082
2212    1049897
321      169602
Name: count, dtype: int64


In [3]:
# ─────────────────────────────────────────────────────────────────────
# What was the MC truth identity of those same tracks?
# Column "mc_matching_pid" is the PDG code of the MC particle that
# geometrically matched the reconstructed track (|Δφ| < 6°, |Δθ| < 2°).
# -9999 means no MC particle matched (rare, <5%).
# Many entries will be non-pion/non-kaon truths — protons, photons,
# decay products of ρ⁰/ω/η, etc. — that EB mis-identified as a pion/kaon.
# ─────────────────────────────────────────────────────────────────────
print("\nMC-truth PID (top 15):")
print(df["mc_matching_pid"].value_counts().head(15))
print(f"\nUnmatched tracks (no MC match): {(df['mc_matching_pid'] == -9999).sum()}")




MC-truth PID (top 15):
mc_matching_pid
 211     1448414
 2212     991608
-9999     147312
 321      110178
 11         7311
 22         7281
-211        5994
 2112       4798
-11          971
-321         350
 130         267
-2212         48
-13           38
-2112          9
-12            2
Name: count, dtype: int64

Unmatched tracks (no MC match): 147312


In [4]:
# ─────────────────────────────────────────────────────────────────────
# THE KEY DIAGNOSTIC for this project — the contamination matrix.
# Rows = what EB called the track. Columns = what it actually was.
# Off-diagonal entries are the misidentifications ML must learn to fix.
# For Cooper's project the most important row is EB pid=321 (EB-K+):
# how many of those are really K+ (diagonal) vs π+ contaminating (off-diagonal)?
# ─────────────────────────────────────────────────────────────────────
# Show only the most common truth species to keep the table readable
top_truths = df["mc_matching_pid"].value_counts().head(8).index
contam = pd.crosstab(
    df["pid"],
    df["mc_matching_pid"].where(df["mc_matching_pid"].isin(top_truths), other="other"),
    margins=True,
)
print("\nContamination matrix (EB pid × MC-truth pid):")
print(contam)




Contamination matrix (EB pid × MC-truth pid):
mc_matching_pid   -9999  -211    11    22      211     321  2112    2212  \
pid                                                                        
211              112486  3444  5079  4795  1359838    6378  4214    7565   
321                4749   166   229   187    61933   99087   256    2938   
2212              30077  2384  2003  2299    26643    4713   328  981105   
All              147312  5994  7311  7281  1448414  110178  4798  991608   

mc_matching_pid  other      All  
pid                              
211               1283  1505082  
321                 57   169602  
2212               345  1049897  
All               1685  2724581  


In [5]:
# ─────────────────────────────────────────────────────────────────────
# Quick sanity stats on key training features.
# - beta:    should be roughly [0.5, 1.0]. Values > 1 are reco artifacts.
# - chi2pid: should mostly be in [-5, +5]. Sentinel 9999 = EB couldn't compute it.
# - rich_RQ: -9999 means no RICH hit (most tracks — RICH covers one sector only).
# ─────────────────────────────────────────────────────────────────────
print("\nKey feature stats:")
print(df[["beta", "chi2pid", "rich_RQ"]].describe())
print(f"chi2pid sentinels (9999): {(df['chi2pid'] == 9999).sum()}")
print(f"Tracks with RICH hit (rich_RQ != -9999): {(df['rich_RQ'] != -9999).sum()}")


Key feature stats:
               beta       chi2pid       rich_RQ
count  2.724581e+06  2.724581e+06  2.724581e+06
mean   9.124036e-01  1.365430e+02 -9.279871e+03
std    8.122819e+00  1.170192e+03  2.583348e+03
min   -5.712954e+03 -1.092393e+05 -9.999000e+03
25%    9.016948e-01 -7.244107e-01 -9.999000e+03
50%    9.796335e-01  7.788338e-02 -9.999000e+03
75%    9.942113e-01  9.062551e-01 -9.999000e+03
max    7.316554e+03  9.999000e+03  1.000000e+00
chi2pid sentinels (9999): 37689
Tracks with RICH hit (rich_RQ != -9999): 195945


In [6]:
matched = df[df["mc_matching_pid"]!=-9999]
print((matched["mc_matching_pid"]).sum())

2543314610
